# Before Agent Guadrail

In [1]:
from dotenv import load_dotenv 
from langchain_groq import ChatGroq
load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b")

In [13]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from typing import Any

class ContentFilterMiddlware(AgentMiddleware):

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = banned_keywords

    @hook_config(can_jump_to="end")
    def before_agent(self, state: AgentState,runtime: Runtime) -> dict[str,Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type !="human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:

            if keyword in content:
                print(f"Blocked - keyword detected: {keyword}")
                return {
                    "messages": [
                        {
                            "role": "assistant",
                            "content": f"Your message contains a banned keyword: {keyword}. Please rephrase your query."
                        }
                    ],
                    "jump_to":"end"
                }
        return None

@tool
def search_tool(query: str)-> str:
    """Search for information"""
    return f"Result for : {query}"


agent = create_agent(
    model=model, 
    tools=[search_tool],
    middleware=[ContentFilterMiddlware(banned_keywords=["banned", "hack","jailbreak","malware"])],
)

In [16]:
response =agent.invoke({
    "messages":[
        {"role":"human","content":"How to learn AI?"}
    ]})

In [19]:
print(response["messages"][-1])

content='## A Practical Road‑Map for Learning Artificial Intelligence  \n\nBelow is a **step‑by‑step learning path** you can follow (or adapt) whether you’re a complete beginner or already have some technical background.  Each stage lists the core concepts, why they matter, recommended resources, and concrete “hands‑on” milestones so you can see progress as you go.\n\n---\n\n### 1️⃣ Build the Foundations  \n\n| Core Area | What to Master | Why It Matters | Starter Resources | Hands‑On Milestone |\n|-----------|----------------|----------------|-------------------|--------------------|\n| **Mathematics** | • Linear Algebra (vectors, matrices, eigenvalues)  <br>• Calculus (derivatives, gradients, chain rule)  <br>• Probability & Statistics (distributions, Bayes, expectation) | All ML algorithms are built on these concepts. | • “Essence of Linear Algebra” (3Blue1Brown YouTube)  <br>• Khan Academy – Linear Algebra, Calculus, Stats  <br>• Book: *Mathematics for Machine Learning* (free PDF) 